# 🚀 EgoBlur Gen2 — Optimized (Batched + GPU + Streaming)

This notebook runs the **optimized EgoBlur Gen2** pipeline with:
- ✅ Batch inference (process N frames per GPU pass)
- ✅ GPU-only preprocessing (no CPU roundtrips)
- ✅ GPU-accelerated blurring
- ✅ Streaming output via `cv2.VideoWriter` (flat RAM, no memory blowup)
- ✅ OpenCV VideoCapture (no MoviePy/ALSA errors)
- ✅ Checkpointing (resume after interruption)

**Runtime:** Select **GPU** under Runtime → Change runtime type → T4/A100/L4

## 1. Setup — Clone repo + install dependencies

In [ ]:
import os
os.chdir('/content')

# Clone the repo
!git clone https://github.com/facebookresearch/EgoBlur.git egoblur 2>/dev/null || echo 'Repo already cloned'
os.chdir('/content/egoblur')

# Install dependencies (skip torch — Colab already has it)
!pip install -q opencv-python-headless>=4.7 tqdm>=4.64 numpy>=1.24

In [ ]:
# Verify GPU is available
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected! Go to Runtime → Change runtime type → GPU')

## 2. Patch — Apply optimizations to the codebase

This writes the optimized files directly into the cloned repo.

In [ ]:
# --------------------------------------------------------------------------
# Patch predictor.py: add pre_process_gpu, compute_resize_dims, remove deepcopy, fix get_detections
# --------------------------------------------------------------------------

PREDICTOR_PATH = '/content/egoblur/gen2/script/predictor.py'

with open(PREDICTOR_PATH, 'r') as f:
    src = f.read()

# 1. Remove 'from copy import deepcopy'
src = src.replace('from copy import deepcopy\n', '')

# 2. Add 'import torch.nn.functional as F' after 'import torch'
src = src.replace('import torch\n', 'import torch\nimport torch.nn.functional as F\n')

# 3. Remove deepcopy call in _post_process
src = src.replace('        preds0 = deepcopy(preds0)\n', '')

# 4. Fix get_detections to append None for empty frames
src = src.replace(
    '            if not boxes.any():\n                continue',
    '            if not boxes.any():\n                detections_batch.append(None)\n                continue'
)

# 5. Add compute_resize_dims and pre_process_gpu methods before 'def pre_process'
gpu_methods = '''
    @staticmethod
    def compute_resize_dims(h, w, short_edge, max_size):
        scale = float(short_edge) / min(h, w)
        new_h = h * scale
        new_w = w * scale
        if max(new_h, new_w) > max_size:
            scale = float(max_size) / max(new_h, new_w)
            new_h *= scale
            new_w *= scale
        return int(new_h + 0.5), int(new_w + 0.5)

'''

pre_process_gpu_method = '''
    def pre_process_gpu(self, bgr_image_batch):
        B, C, H, W = bgr_image_batch.shape
        orig_img_hw_list = [(H, W)] * B
        if self.aug is not None:
            short_edge = self.aug.short_edge_length[0]
            max_size = self.aug.max_size
            new_h, new_w = self.compute_resize_dims(H, W, short_edge, max_size)
            if (new_h, new_w) != (H, W):
                bgr_image_batch = F.interpolate(
                    bgr_image_batch.float(),
                    size=(new_h, new_w),
                    mode="bilinear",
                    align_corners=False,
                ).to(bgr_image_batch.dtype)
            model_input_hw_list = [(new_h, new_w)] * B
        else:
            model_input_hw_list = orig_img_hw_list
        return bgr_image_batch, orig_img_hw_list, model_input_hw_list

'''

src = src.replace(
    '    def pre_process(',
    gpu_methods + '    def pre_process('
)

# Insert pre_process_gpu after the pre_process method's return statement
marker = '        return torch.stack(img_tensor_list), orig_img_hw_list, model_input_hw_list'
src = src.replace(
    marker,
    marker + '\n' + pre_process_gpu_method
)

with open(PREDICTOR_PATH, 'w') as f:
    f.write(src)

print('✅ predictor.py patched')

In [ ]:
# --------------------------------------------------------------------------
# Write the optimized fast script (uses cv2.VideoWriter for streaming)
# --------------------------------------------------------------------------

FAST_SCRIPT = r'''
import argparse, json, math, os, signal, sys, time
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import torch
import torchvision.transforms.functional as TF
from gen2.script.constants import (
    FACE_THRESHOLDS_GEN2, LP_THRESHOLDS_GEN2, RESIZE_MAX_GEN2, RESIZE_MIN_GEN2,
)
from gen2.script.detectron2.export.torchscript_patch import patch_instances
from gen2.script.predictor import ClassID, EgoblurDetector, PATCH_INSTANCES_FIELDS
from gen2.script.utils import get_device, setup_logger, validate_inputs
from tqdm.auto import tqdm

logger = setup_logger()


def _get_threshold(camera_name, user_threshold, threshold_map):
    if user_threshold is not None:
        return user_threshold
    if threshold_map is not None:
        if camera_name is not None:
            return threshold_map.get(camera_name, threshold_map["camera-rgb"])
        return threshold_map["camera-rgb"]
    raise ValueError("Cannot resolve score threshold.")


def _scale_box(box, max_width, max_height, scale):
    x1, y1, x2, y2 = box[0], box[1], box[2], box[3]
    w, h = x2 - x1, y2 - y1
    xc, yc = x1 + w / 2, y1 + h / 2
    w, h = scale * w, scale * h
    return [max(xc - w/2, 0), max(yc - h/2, 0), min(xc + w/2, max_width), min(yc + h/2, max_height)]


def blur_regions_gpu(image_tensor, detections, scale_factor):
    if not detections:
        return image_tensor
    C, H, W = image_tensor.shape
    ksize = max(H // 2, 1) | 1
    if ksize % 2 == 0:
        ksize += 1
    sigma = ksize / 3.0
    img_float = image_tensor.unsqueeze(0).float()
    blurred = TF.gaussian_blur(img_float, kernel_size=[ksize, ksize], sigma=[sigma, sigma])
    blurred = blurred.squeeze(0).to(image_tensor.dtype)
    mask = np.zeros((H, W), dtype=np.uint8)
    for box in detections:
        if scale_factor != 1.0:
            box = _scale_box(box, W, H, scale_factor)
        x1, y1, x2, y2 = int(box[0]), int(box[1]), int(box[2]), int(box[3])
        w, h = x2 - x1, y2 - y1
        if w <= 0 or h <= 0:
            continue
        cv2.ellipse(mask, (((x1+x2)//2, (y1+y2)//2), (w, h), 0), 255, -1)
    mask_tensor = torch.from_numpy(mask).to(image_tensor.device).bool()
    return torch.where(mask_tensor.unsqueeze(0), blurred, image_tensor)


def _checkpoint_path(p):
    return p + ".checkpoint.json"

def load_checkpoint(p):
    cp = _checkpoint_path(p)
    if os.path.exists(cp):
        with open(cp) as f:
            return json.load(f)
    return None

def save_checkpoint(p, frames_written, total_frames, fps):
    with open(_checkpoint_path(p), "w") as f:
        json.dump({"frames_written": frames_written, "total_frames": total_frames, "fps": fps}, f)

def delete_checkpoint(p):
    cp = _checkpoint_path(p)
    if os.path.exists(cp):
        os.remove(cp)


class StreamingVideoWriter:
    """Write BGR frames one-at-a-time via cv2.VideoWriter. Zero RAM accumulation."""
    def __init__(self, output_path, width, height, fps, append=False):
        self.output_path = output_path
        self.width, self.height, self.fps = width, height, fps
        if append:
            self.temp_path = output_path + ".resume_part.mp4"
            target = self.temp_path
        else:
            self.temp_path = None
            target = output_path
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        self.writer = cv2.VideoWriter(target, fourcc, fps, (width, height))
        if not self.writer.isOpened():
            raise RuntimeError(f"cv2.VideoWriter failed to open: {target}")
        self.frames_written = 0

    def write_frame(self, bgr_frame):
        if bgr_frame.dtype != np.uint8:
            bgr_frame = np.clip(bgr_frame, 0, 255).astype(np.uint8)
        self.writer.write(np.ascontiguousarray(bgr_frame))
        self.frames_written += 1

    def close(self):
        self.writer.release()
        if self.temp_path and os.path.exists(self.temp_path):
            import subprocess
            concat_path = self.output_path + ".concat_list.txt"
            with open(concat_path, "w") as f:
                f.write(f"file \'{os.path.abspath(self.output_path)}\'\n")
                f.write(f"file \'{os.path.abspath(self.temp_path)}\'\n")
            merged = self.output_path + ".merged.mp4"
            subprocess.run(["ffmpeg","-y","-f","concat","-safe","0","-i",concat_path,"-c","copy",merged],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
            os.replace(merged, self.output_path)
            os.remove(self.temp_path)
            os.remove(concat_path)


def run_egoblur_fast(
    input_video_path,
    output_video_path,
    face_model_path=None,
    lp_model_path=None,
    camera_name=None,
    face_score_threshold=None,
    lp_score_threshold=None,
    nms_iou_threshold=0.5,
    scale_factor_detections=1.0,
    batch_size=8,
    checkpoint_interval=100,
    no_resume=False,
):
    """Main entry point for optimized EgoBlur video processing."""
    if face_model_path is None and lp_model_path is None:
        raise ValueError("Provide at least one of face_model_path or lp_model_path")

    device = get_device()
    face_threshold = _get_threshold(camera_name, face_score_threshold, FACE_THRESHOLDS_GEN2)
    lp_threshold = _get_threshold(camera_name, lp_score_threshold, LP_THRESHOLDS_GEN2)

    face_detector = None
    if face_model_path is not None:
        face_detector = EgoblurDetector(
            model_path=face_model_path, device=device, detection_class=ClassID.FACE,
            score_threshold=face_threshold, nms_iou_threshold=nms_iou_threshold,
            resize_aug={"min_size_test": RESIZE_MIN_GEN2, "max_size_test": RESIZE_MAX_GEN2},
        )

    lp_detector = None
    if lp_model_path is not None:
        lp_detector = EgoblurDetector(
            model_path=lp_model_path, device=device, detection_class=ClassID.LICENSE_PLATE,
            score_threshold=lp_threshold, nms_iou_threshold=nms_iou_threshold,
            resize_aug={"min_size_test": RESIZE_MIN_GEN2, "max_size_test": RESIZE_MAX_GEN2},
        )

    # Open video
    cap = cv2.VideoCapture(input_video_path)
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {input_video_path}")
    input_fps = cap.get(cv2.CAP_PROP_FPS)
    if not (isinstance(input_fps, (int, float)) and math.isfinite(input_fps) and input_fps > 0):
        cap.release()
        raise ValueError(f"Invalid FPS for {input_video_path}")
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = float(input_fps)

    # Checkpointing
    start_frame, append_mode = 0, False
    if not no_resume:
        cp = load_checkpoint(output_video_path)
        if cp is not None and os.path.exists(output_video_path):
            start_frame = cp["frames_written"]
            if start_frame >= total_frames:
                print("All frames already processed.")
                cap.release()
                delete_checkpoint(output_video_path)
                return
            print(f"Resuming from frame {start_frame}/{total_frames} ({start_frame/total_frames*100:.1f}%)")
            append_mode = True
    if start_frame > 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    os.makedirs(os.path.dirname(output_video_path) or '.', exist_ok=True)
    writer = StreamingVideoWriter(output_video_path, W, H, fps, append=append_mode)

    total_inference_time = 0.0
    total_blur_time = 0.0
    total_frame_time = 0.0
    frames_processed = 0
    global_frame_idx = start_frame

    interrupted = False
    def sig_handler(sig, frame):
        nonlocal interrupted
        interrupted = True
    prev_handler = signal.getsignal(signal.SIGINT)
    signal.signal(signal.SIGINT, sig_handler)

    remaining = total_frames - start_frame
    pbar = tqdm(total=remaining, desc="Processing frames", unit="frame")

    try:
        with patch_instances(fields=PATCH_INSTANCES_FIELDS):
            while not interrupted:
                batch_bgr = []
                for _ in range(batch_size):
                    ret, frame = cap.read()
                    if not ret:
                        break
                    if frame.ndim == 2:
                        frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
                    batch_bgr.append(frame)
                if not batch_bgr:
                    break

                t0 = time.time()
                B = len(batch_bgr)
                batch_np = np.stack(batch_bgr, axis=0)
                batch_tensor = torch.from_numpy(batch_np.transpose(0, 3, 1, 2)).to(device)

                all_dets = [[] for _ in range(B)]

                if face_detector is not None:
                    img_b, orig_hw, model_hw = face_detector.pre_process_gpu(batch_tensor)
                    t_inf = time.time()
                    preds = face_detector.inference(img_b)
                    total_inference_time += time.time() - t_inf
                    det_batch = face_detector.get_detections(
                        output_tensor=preds, timestamp_s=0.0, stream_id="",
                        rotation_angle=0.0, model_input_hw_list=model_hw, target_img_hw_list=orig_hw,
                    )
                    for i, det in enumerate(det_batch):
                        if det is not None:
                            all_dets[i].extend(det.face_bboxes.tolist())

                if lp_detector is not None:
                    img_b, orig_hw, model_hw = lp_detector.pre_process_gpu(batch_tensor)
                    t_inf = time.time()
                    preds = lp_detector.inference(img_b)
                    total_inference_time += time.time() - t_inf
                    det_batch = lp_detector.get_detections(
                        output_tensor=preds, timestamp_s=0.0, stream_id="",
                        rotation_angle=0.0, model_input_hw_list=model_hw, target_img_hw_list=orig_hw,
                    )
                    for i, det in enumerate(det_batch):
                        if det is not None:
                            all_dets[i].extend(det.lp_bboxes.tolist())

                t_blur = time.time()
                for i in range(B):
                    result = blur_regions_gpu(batch_tensor[i], all_dets[i], scale_factor_detections)
                    writer.write_frame(result.cpu().numpy().transpose(1, 2, 0))
                total_blur_time += time.time() - t_blur

                total_frame_time += time.time() - t0
                frames_processed += B
                global_frame_idx += B
                pbar.update(B)

                if frames_processed % checkpoint_interval < batch_size:
                    save_checkpoint(output_video_path, global_frame_idx, total_frames, fps)

    except Exception:
        save_checkpoint(output_video_path, global_frame_idx, total_frames, fps)
        raise
    finally:
        pbar.close()
        signal.signal(signal.SIGINT, prev_handler)
        cap.release()
        writer.close()

    if interrupted:
        save_checkpoint(output_video_path, global_frame_idx, total_frames, fps)
        print(f"Interrupted at frame {global_frame_idx}/{total_frames}. Run again to resume.")
        return

    delete_checkpoint(output_video_path)

    if frames_processed > 0:
        achieved_fps = frames_processed / total_frame_time if total_frame_time > 0 else 0
        print("\n" + "=" * 60)
        print("SPEED REPORT")
        print("=" * 60)
        print(f"Frames processed: {frames_processed}")
        print(f"Batch size:       {batch_size}")
        print(f"Inference:        {total_inference_time/frames_processed:.4f} s/frame")
        print(f"Blurring:         {total_blur_time/frames_processed:.4f} s/frame")
        print(f"Total:            {total_frame_time/frames_processed:.4f} s/frame")
        print(f"Effective FPS:    {achieved_fps:.2f}")
        print("-" * 60)
        print(f"Total time:       {total_frame_time:.1f} seconds")
        print("=" * 60)
    print(f"\n✅ Output saved to: {output_video_path}")
'''

fast_path = '/content/egoblur/gen2/script/demo_ego_blur_gen2_fast.py'
with open(fast_path, 'w') as f:
    f.write(FAST_SCRIPT)
print('✅ demo_ego_blur_gen2_fast.py written')

## 3. Upload Model

Upload your `.jit` model file, or copy from Google Drive.

In [ ]:
# Option A: Upload from your machine
from google.colab import files
print('Upload your ego_blur_face_gen2.jit file:')
uploaded = files.upload()
for name in uploaded:
    os.rename(name, f'/content/egoblur/{name}')
    print(f'  → /content/egoblur/{name}')

In [ ]:
# Option B: Copy from Google Drive (uncomment and edit path)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp '/content/drive/MyDrive/path/to/ego_blur_face_gen2.jit' /content/egoblur/

## 4. Upload your input video

In [ ]:
# Option A: Upload from your machine
from google.colab import files
print('Upload your input video:')
uploaded = files.upload()
INPUT_VIDEO = None
for name in uploaded:
    INPUT_VIDEO = f'/content/egoblur/{name}'
    os.rename(name, INPUT_VIDEO)
    print(f'  → {INPUT_VIDEO}')

In [ ]:
# Option B: Copy from Google Drive (uncomment and edit)
# INPUT_VIDEO = '/content/egoblur/my_video.mp4'
# !cp '/content/drive/MyDrive/path/to/video.mp4' "$INPUT_VIDEO"

## 5. 🚀 Run EgoBlur Fast!

Adjust `BATCH_SIZE` based on your GPU VRAM:
- **T4 (16GB):** 4-8
- **L4 (24GB):** 6-12
- **A100 (40GB):** 8-16

In [ ]:
# ── Configure ─────────────────────────────────────────────────
FACE_MODEL = '/content/egoblur/ego_blur_face_gen2.jit'
# LP_MODEL = '/content/egoblur/ego_blur_lp_gen2.jit'  # uncomment if using LP model
# INPUT_VIDEO = '/content/egoblur/my_video.mp4'  # set if not uploaded above
OUTPUT_VIDEO = '/content/egoblur/output_blurred.mp4'
BATCH_SIZE = 8
# ──────────────────────────────────────────────────────────────

import sys
os.chdir('/content/egoblur')
sys.path.insert(0, '/content/egoblur')

# Clear GPU cache before starting
torch.cuda.empty_cache()

from gen2.script.demo_ego_blur_gen2_fast import run_egoblur_fast

run_egoblur_fast(
    input_video_path=INPUT_VIDEO,
    output_video_path=OUTPUT_VIDEO,
    face_model_path=FACE_MODEL,
    # lp_model_path=LP_MODEL,  # uncomment for license plate detection
    batch_size=BATCH_SIZE,
    checkpoint_interval=100,
)

## 6. Download output

In [ ]:
# Option A: Download directly
from google.colab import files
files.download(OUTPUT_VIDEO)

In [ ]:
# Option B: Copy to Google Drive
# !cp "$OUTPUT_VIDEO" '/content/drive/MyDrive/path/to/output/'

## 7. (Optional) Compare with original speed

In [ ]:
# Run original (slow) version for comparison
!cd /content/egoblur && pip install -q 'moviepy>=1.0.3,<2.0' 2>/dev/null
!cd /content/egoblur && python -m gen2.script.demo_ego_blur_gen2 \
    --face_model_path "$FACE_MODEL" \
    --input_video_path "$INPUT_VIDEO" \
    --output_video_path /content/egoblur/output_original.mp4